# generator-project-and-reshape — worked example 1: Project a latent vector to a 7x7 spatial seed

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-project-and-reshape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The first layer of a DCGAN generator turns a flat noise vector into a small multi-channel feature map. A `Linear(latent_dim, C*H*W)` projects the latent, then a `view(B, C, H, W)` reshapes the flat output into a spatial seed ready for transposed convolutions. The 7x7 seed is the MNIST-scale choice (two 2x upsamples land at 28x28).

## Worked solution

We project `z: (B, latent_dim)` into a `(B, C, 7, 7)` seed.

1. The affine projection is `z @ weight.T + bias`, where `weight` is `(C*7*7, latent_dim)` and `bias` is `(C*7*7,)`. This is exactly what `nn.Linear(latent_dim, C*7*7)` computes; we do it manually to see the shapes.
2. The result `flat` has shape `(B, C*7*7)` — one long vector per batch element.
3. We reshape with `flat.view(B, C, 7, 7)`. The total element count is unchanged; `view` just reinterprets the flat features as a 7x7 map across `C` channels.
4. We print the seed shape and confirm the channels actually differ (a degenerate projection would give identical channels), which would defeat the spatial seed's purpose.

In [ ]:
import torch as t

t.manual_seed(0)
latent_dim, C = 100, 16
z = t.randn(4, latent_dim)
weight = t.randn(C * 7 * 7, latent_dim) * 0.02
bias = t.zeros(C * 7 * 7)

def project_seed(z, weight, bias, C, spatial):
    B = z.shape[0]
    flat = z @ weight.T + bias
    return flat.view(B, C, spatial, spatial)

seed = project_seed(z, weight, bias, C, 7)
print(seed.shape)
print('channels differ:', bool(not t.allclose(seed[0, 0], seed[0, 1])))